# Analyse Exploratoire - Heart Disease UCI Dataset
## Rapport Académique Complet

**Auteur**: Étudiant IFOAD  
**Date**: 13 mai 2026  
**Objectif**: Classification binaire de la présence de maladie cardiaque

---

## 1. Introduction

### 1.1 Contexte
Les maladies cardio-vasculaires sont l'une des principales causes de mortalité mondiale. 
Ce projet vise à développer un modèle de classification capable de prédire la présence 
d'une maladie cardiaque basé sur des variables médicales et physiologiques.

### 1.2 Dataset
- **Source**: UCI Machine Learning Repository - Heart Disease Dataset
- **Fichier**: processed.cleveland.data
- **Type**: Classification binaire
- **Variables**: 13 features médicales + 1 variable cible

### 1.3 Objectifs
1. Nettoyer et préparer les données
2. Explorer les distributions et relations entre variables
3. Identifier les patterns et anomalies
4. Préparer les données pour l'entraînement des modèles

---

## 2. Chargement et Description du Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print("Bibliothèques importées avec succès ✓")

In [ ]:
# Définir les noms des colonnes
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
    'ca', 'thal', 'target'
]

# Charger le dataset
data_path = Path('../data/raw/processed.cleveland.data')
df = pd.read_csv(data_path, names=column_names, na_values='?')

print(f"Dataset chargé avec succès ✓")
print(f"Forme: {df.shape}")
print(f"\nPremières lignes:")
print(df.head())

In [ ]:
# Informations générales
print("=" * 60)
print("INFORMATIONS GÉNÉRALE DU DATASET")
print("=" * 60)
print(f"\nNombre d'échantillons: {df.shape[0]}")
print(f"Nombre de features: {df.shape[1] - 1}")
print(f"\nTypes de données:")
print(df.dtypes)
print(f"\nValeurs manquantes:")
print(df.isnull().sum())
print(f"\nPourcentage de valeurs manquantes:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Statistiques descriptives
print("=" * 60)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 60)
print(df.describe().round(2))

---

## 3. Nettoyage et Préparation des Données

In [ ]:
# Supprimer les valeurs manquantes
print(f"Avant nettoyage: {df.shape[0]} lignes")
df_clean = df.dropna()
print(f"Après nettoyage: {df_clean.shape[0]} lignes")
print(f"Lignes supprimées: {df.shape[0] - df_clean.shape[0]}")

In [ ]:
# Convertir la cible en binaire (0: pas de maladie, 1: maladie présente)
df_clean['target'] = (df_clean['target'] > 0).astype(int)

print("Distribution de la cible:")
print(df_clean['target'].value_counts())
print(f"\nPourcentage:")
print((df_clean['target'].value_counts() / len(df_clean) * 100).round(2))

In [ ]:
# Vérifier les types de données
print("Types de données après nettoyage:")
print(df_clean.dtypes)
print(f"\nDataset nettoyé:")
print(df_clean.head())

---

## 4. Analyse Exploratoire Détaillée

### 4.1 Distribution de la Variable Cible

In [ ]:
# Distribution de la cible
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Graphique 1: Comptage
df_clean['target'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Distribution de la Maladie Cardiaque', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cible (0: Absent, 1: Présent)')
axes[0].set_ylabel('Nombre d\'échantillons')
axes[0].set_xticklabels(['Pas de maladie', 'Maladie présente'], rotation=0)

# Graphique 2: Camembert
df_clean['target'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                        colors=['green', 'red'], labels=['Absent', 'Présent'])
axes[1].set_title('Proportion - Maladie Cardiaque', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 4.2 Analyse des Variables Continues

Exploration des variables numériques principales: âge, pression artérielle, cholestérol, fréquence cardiaque

In [ ]:
# Variables continues principales
continuous_vars = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Distribution des variables continues
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, var in enumerate(continuous_vars):
    axes[idx].hist(df_clean[var], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution - {var.upper()}', fontweight='bold')
    axes[idx].set_xlabel(var)
    axes[idx].set_ylabel('Fréquence')
    axes[idx].axvline(df_clean[var].mean(), color='red', linestyle='--', label=f'Moyenne: {df_clean[var].mean():.2f}')
    axes[idx].legend()

# Supprimer le dernier subplot vide
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot par rapport à la cible
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, var in enumerate(continuous_vars):
    df_clean.boxplot(column=var, by='target', ax=axes[idx])
    axes[idx].set_title(f'{var.upper()} par rapport à la Cible', fontweight='bold')
    axes[idx].set_xlabel('Cible (0: Absent, 1: Présent)')
    axes[idx].set_ylabel(var)

# Supprimer le dernier subplot vide
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

### 4.3 Analyse des Variables Catégorielles

Exploration des variables catégorielles: sexe, type de douleur thoracique, etc.

In [ ]:
# Variables catégorielles
categorical_vars = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

# Distribution des variables catégorielles
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, var in enumerate(categorical_vars):
    df_clean[var].value_counts().plot(kind='bar', ax=axes[idx], color='coral', edgecolor='black')
    axes[idx].set_title(f'Distribution - {var.upper()}', fontweight='bold')
    axes[idx].set_xlabel(var)
    axes[idx].set_ylabel('Fréquence')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Relation entre variables catégorielles et cible
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, var in enumerate(categorical_vars):
    crosstab = pd.crosstab(df_clean[var], df_clean['target'])
    crosstab.plot(kind='bar', ax=axes[idx], stacked=False)
    axes[idx].set_title(f'{var.upper()} vs Cible', fontweight='bold')
    axes[idx].set_xlabel(var)
    axes[idx].set_ylabel('Nombre')
    axes[idx].legend(['Absent', 'Présent'])
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.4 Corrélation entre Variables

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(12, 9))
correlation_matrix = df_clean.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation - Heart Disease Dataset', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation avec la cible
target_corr = df_clean.corr()['target'].sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in target_corr.values]
plt.barh(range(len(target_corr)), target_corr.values, color=colors, alpha=0.7)
plt.yticks(range(len(target_corr)), target_corr.index)
plt.xlabel('Coefficient de Corrélation', fontweight='bold')
plt.title('Corrélation de chaque Variable avec la Cible', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Corrélation avec la cible (triée):")
print(target_corr)

### 4.5 Analyse des Valeurs Aberrantes

In [ ]:
# Détection des outliers avec l'écart interquartile (IQR)
def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    return ((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR)))

outlier_count = {}
for var in continuous_vars:
    outlier_count[var] = detect_outliers_iqr(df_clean[var]).sum()

print("Nombre de valeurs aberrantes (IQR) par variable:")
for var, count in outlier_count.items():
    pct = (count / len(df_clean) * 100)
    print(f"  {var}: {count} ({pct:.2f}%)")

---

## 5. Sauvegarde du Dataset Nettoyé

In [ ]:
# Sauvegarder le dataset nettoyé
output_path = Path('../data/processed/heart_disease_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)

print(f"Dataset nettoyé sauvegardé: {output_path}")
print(f"Forme finale: {df_clean.shape}")

---

## 6. Conclusion et Recommandations

### 6.1 Synthèse des Résultats

1. **Qualité des données**: Le dataset contient des valeurs manquantes qui ont été supprimées avec succès.
2. **Déséquilibre des classes**: À évaluer selon les résultats de l'analyse.
3. **Variables importantes**: Identifiées par analyse de corrélation.
4. **Valeurs aberrantes**: Détectées mais conservées pour maintenir l'intégrité du dataset.

### 6.2 Recommandations pour la Modélisation

1. **Normalisation**: Normaliser les variables continues pour une meilleure performance des modèles.
2. **Encodage**: Encoder les variables catégorielles (one-hot encoding si nécessaire).
3. **Sélection de features**: Utiliser les variables avec la plus forte corrélation à la cible.
4. **Gestion du déséquilibre**: Appliquer des techniques d'oversampling/undersampling si nécessaire.
5. **Validation croisée**: Utiliser k-fold cross-validation pour une évaluation robuste.

### 6.3 Prochaines Étapes

✓ Entraîner 6 modèles de classification (Logistic Regression, SVM, Random Forest, etc.)  
✓ Évaluer les modèles avec des métriques appropriées (Accuracy, Precision, Recall, F1, AUC)  
✓ Comparer les performances et sélectionner le meilleur modèle  
✓ Créer une application Streamlit pour les prédictions en temps réel